In [ ]:
import pandas as pd, numpy as np, re, os

DATA_DIR  = "../data/"
OUT_DIR   = "../data/cleaned"
FILES = {
    "brecorder": "brecorder_news_ubl.csv",
    "mettis":    "mettis_global_news_ubl.csv",
    "profit":    "profitpktoday_news_ubl.csv",
}
START_DATE = "2021-01-01"
MIN_WORDS = 30
CLOSE_TIME = "15:30"       # PSX close 
DATE_ONLY_POLICY = "next"     # no time known: "next" = safe (no leakage)
COMPANY_REGEX = r"\bUBL\b|United Bank"
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# 1. Load + standardize
frames = []
for src, fn in FILES.items():
    d = pd.read_csv(os.path.join(DATA_DIR, fn))
    d["source"] = src
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
df = df.rename(columns={"snippet": "text", "published": "published_raw"})
print(df.groupby("source").size())

source
brecorder    111
mettis       243
profit        70
dtype: int64


In [ ]:
# 2. Parse dates -> PKT.... Brecorder has a timezone + time; the others are date-only.
raw = df["published_raw"].astype(str).str.strip()
df["has_time"] = raw.str.len() > 19   

ts = pd.to_datetime(raw.where(df["has_time"]), errors="coerce", utc=True).dt.tz_convert("Asia/Karachi")
day = pd.to_datetime(raw.str[:10], errors="coerce")
df["pub_date"] = day                       # calendar date as published
df["pub_ts"]= ts                        # PKT timestamp (NaT when only a date is known)
print("unparseable dates:", df["pub_date"].isna().sum())
print(df.groupby("source")["has_time"].mean().round(2))

unparseable dates: 0
source
brecorder    1.0
mettis       0.0
profit       0.0
Name: has_time, dtype: float64


In [6]:
# 3. Clean text
def clean(t):
    t = "" if pd.isna(t) else str(t)
    t = re.sub(r"<[^>]+>", " ", t)    # html
    t = re.sub(r"http\S+", " ", t) 
    t = re.sub(r"^\s*[A-Z][a-z]+ \d{1,2}, \d{4} \(MLN\)\s*:\s*", "", t)  # Mettis dateline
    t = re.sub(r"(^|\n)\s*-\s+", r"\1", t)       # Brecorder bullet lead-ins
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text"]  = df["text"].map(clean)
df["title"] = df["title"].map(clean)
df["n_words"] = df["text"].str.split().str.len()
df["mentions_company"] = (df["title"] + " " + df["text"]).str.contains(COMPANY_REGEX, case=False, regex=True)
print(df["n_words"].describe().round(0))
print("share not mentioning UBL:", (~df["mentions_company"]).mean().round(3))

count     424.0
mean      390.0
std       312.0
min        45.0
25%       196.0
50%       320.0
75%       477.0
max      3215.0
Name: n_words, dtype: float64
share not mentioning UBL: 0.061


In [7]:
# 4. Filter: date window, minimum length
n0 = len(df)
df = df[(df["pub_date"] >= START_DATE) & (df["n_words"] >= MIN_WORDS)].copy()
print(f"kept {len(df)} of {n0} (dropped pre-{START_DATE} and <{MIN_WORDS} words)")

kept 315 of 424 (dropped pre-2021-01-01 and <30 words)


In [8]:
# 5. Deduplicate: exact URL, then normalized title.
df["title_key"] = df["title"].str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
df = df.drop_duplicates("url")
df = df.sort_values("n_words", ascending=False)
srcs = df.groupby("title_key")["source"].apply(lambda s: ",".join(sorted(set(s)))).rename("all_sources")
df = df.drop_duplicates("title_key").merge(srcs, on="title_key")
print("after dedupe:", len(df))
print(df["all_sources"].value_counts().head())

after dedupe: 277
all_sources
mettis       134
brecorder    101
profit        42
Name: count, dtype: int64


In [10]:
# 6. Trading-day alignment ( rule: news after market close counts for the NEXT trading day)
from pandas.tseries.offsets import BDay
close = pd.Timedelta(hours=int(CLOSE_TIME[:2]), minutes=int(CLOSE_TIME[3:]))

def feature_date(r):
    d = r["pub_date"]
    if r["has_time"] and pd.notna(r["pub_ts"]):
        after_close = (r["pub_ts"] - r["pub_ts"].normalize()) >= close
    else:
        after_close = (DATE_ONLY_POLICY == "next")
    if d.weekday() >= 5:                       # weekend -> next Monday
        return d + BDay(1)
    return d + BDay(1) if after_close else d

df["feature_date"] = df.apply(feature_date, axis=1)
print(df[["source", "pub_ts", "pub_date", "feature_date"]].sample(8, random_state=0))

        source                    pub_ts   pub_date feature_date
218     mettis                       NaT 2024-01-05   2024-01-08
225     mettis                       NaT 2024-11-06   2024-11-07
190     mettis                       NaT 2023-09-12   2023-09-13
15      mettis                       NaT 2025-07-01   2025-07-02
55      mettis                       NaT 2021-07-16   2021-07-19
198  brecorder 2023-07-20 03:12:28+05:00 2023-07-20   2023-07-20
253     mettis                       NaT 2022-02-08   2022-02-09
81      mettis                       NaT 2021-03-08   2021-03-09


In [11]:
# 7. Final dataset 
df = df.sort_values("feature_date").reset_index(drop=True)
df["article_id"] = df.index
out = df[["article_id", "ticker", "source", "all_sources", "pub_ts", "pub_date", "feature_date",
          "title", "text", "url", "n_words", "mentions_company", "has_time"]]
out.to_csv(os.path.join(OUT_DIR, "ubl_news_clean.csv"), index=False)

print("Total:", len(out), "| range:", out.pub_date.min().date(), "->", out.pub_date.max().date())
print(pd.crosstab(out.pub_date.dt.year, out.source, margins=True))
print("Distinct feature days with news:", out.feature_date.nunique())

Total: 277 | range: 2021-01-26 -> 2026-09-22
source    brecorder  mettis  profit  All
pub_date                                
2021              7      15       2   24
2022              9      21       1   31
2023             12      16       6   34
2024             12      38      12   62
2025              9      22       7   38
2026             52      22      14   88
All             101     134      42  277
Distinct feature days with news: 201
